# 04 - Grad-CAM explainability, subgroup fairness, missing-data audit

**For education only. Not for clinical use.** Nothing here is a diagnosis.

Three questions, in the order a reviewer will ask them:

1. **Does the classifier look at lungs, or at a marker in the corner?** A 0.80 AUROC is worthless if the
   model is reading "Portable" text stamps - notebook 02 measured a -0.061 correlation between image
   brightness and the label, so shortcut learning is a live risk, not a hypothetical one. Grad-CAM maps
   are shown *and* counted (`in_body_fraction`, centroid, lateral bias).
2. **Does it work equally well for every subgroup?** AUROC by sex and age band with bootstrap 95% CIs,
   plus the attention-focus statistics per subgroup. A gap is reported, not smoothed over.
3. **What is missing and what did we do about it?** Renders that failed to decode, absent `PatientSex` /
   `PatientAge` (and `PatientPosition` / `WindowCenter`, when the metadata source is the cached header
   scan rather than the split CSVs), and the policy applied to each.

Needs notebook 03's outputs: `densenet121_lungopacity.pt`, `densenet_metrics.json`, and the split CSVs
that ship in `repo.zip`. Runs on Kaggle or Colab (`RUN_ON`), and the small-`n` paths also run on a CPU
laptop, which is how this notebook was written and checked.

In [ ]:
import os, sys, time, json, platform, random

print("python", platform.python_version(), "|", platform.platform())
print("host guess:", "kaggle" if os.path.isdir("/kaggle") else ("colab" if "/content" in os.getcwd() or os.path.isdir("/content") else "local"))

In [ ]:
# Kaggle images ship pydicom and scikit-learn and have no internet; Colab needs the install.
try:
    import pydicom, sklearn
    print("pydicom", pydicom.__version__, "| scikit-learn", sklearn.__version__, "- already in the image")
except ImportError:
    import subprocess
    print("installing pydicom + scikit-learn ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
                    "pydicom", "scikit-learn"], check=True)

import numpy as np
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| torchvision", __import__("torchvision").__version__)

## 0. Configuration

The attention maps are the deliverable here, so the preprocessing has to be the model's own - both
`make_transform` and `ChestXrayDataset` are imported from `src/data/` instead of being re-typed in this
notebook. `n_predict` is the knob that decides how much work this is: the full test split (4003) on a
GPU, or a few hundred images on a laptop.

In [ ]:
RUN_ON = "kaggle" if os.path.isdir("/kaggle") else ("colab" if os.path.isdir("/content") else "local")

PATHS = {
    # Scratch goes to the ephemeral SSD; notebook 03's checkpoint and metrics are *inputs* here, so they
    # are searched for rather than assumed to be in the output folder.
    "colab": {"scratch": "/content/nb04", "deliverables": "/content/drive/MyDrive/aiml02/results"},
    "kaggle": {"scratch": "/tmp/nb04", "deliverables": "/kaggle/working"},
    "local": {"scratch": os.path.join(os.getcwd(), "_nb04_scratch"),
              "deliverables": os.path.join(os.getcwd(), "_nb04_scratch", "results")},
}[RUN_ON]

CONFIG = {
    "run_on": RUN_ON,
    "scratch": PATHS["scratch"],
    "out_dir": PATHS["deliverables"],
    "cache_root": os.path.join(PATHS["scratch"], "cache"),   # 448 px renders, rebuilt here if absent
    "images_dir": os.path.join(PATHS["scratch"], "stage_2_train_images"),
    "zip_name": "stage_2_train_images.zip",
    "headers_csv": "",           # filled in by the metadata cell if a pre-scanned copy is found
    "cache_size": 448,           # notebook 03's render size; overlays are drawn at this resolution
    "train_size": 224,           # what the network sees
    "batch_eval": 32,
    "n_predict": 4003,           # full test split on a GPU
    "n_focus": 300,              # images whose Grad-CAM statistics are computed (hooks are not free)
    "n_overlay": 9,
    "seed": 7,
    "target_layer": "features.denseblock4",
    "bootstrap": 500,
    "age_bands": [(0, 40), (40, 60), (60, 80), (80, 200)],   # the rows docs/model_card.md reports
}

_LOCAL_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))   # the repository folder

random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
os.makedirs(CONFIG["scratch"], exist_ok=True)
os.makedirs(CONFIG["cache_root"], exist_ok=True)
os.makedirs(CONFIG["out_dir"], exist_ok=True)
print(json.dumps(CONFIG, indent=2))

In [ ]:
import glob

REQUIRE_GPU = CONFIG["n_predict"] > 600
if torch.cuda.is_available():
    device = "cuda"
    print("GPU:", torch.cuda.get_device_name(0),
          "| free VRAM:", round(torch.cuda.mem_get_info()[0] / 2**30, 1), "GB")
elif REQUIRE_GPU:
    raise AssertionError(
        f"n_predict={CONFIG['n_predict']} needs an accelerator. Kaggle: sidebar > Accelerator > GPU T4. "
        "Colab: Runtime > Change runtime type > T4. To check the code on a CPU laptop instead, set "
        "CONFIG['n_predict'] = 200 and CONFIG['n_focus'] = 40.")
else:
    device = "cpu"
    print("running on CPU - only valid for the reduced n_predict set in CONFIG")

## 1. Get the code, the checkpoint and the pixels

Same policy as notebook 03: look in several places and print what was found, because guessing a mount
path costs a wasted 10-minute session.

In [ ]:
import shutil, subprocess, zipfile

EXTRACT_TO = os.path.join(CONFIG["scratch"], "repo_src")
CODE_MARKER = os.path.join("src", "models", "gradcam.py")
CANDIDATES = ["/content/repo.zip", "/kaggle/working/repo.zip",
              "/content/drive/MyDrive/repo.zip", "/content/drive/MyDrive/aiml02/repo.zip",
              "/content/drive/MyDrive/Colab Notebooks/repo.zip",
              os.path.join(os.getcwd(), "..", "repo.zip")]
CANDIDATES += sorted(glob.glob("/kaggle/input/**/repo.zip", recursive=True))
# Kaggle unzips a zip uploaded to a dataset, so the attached code archive can also arrive as a plain tree
# at /kaggle/input/<dataset>/ instead of <dataset>/repo.zip. Look for the marker file that way too.
_MARKER_GLOB = CODE_MARKER.replace(os.sep, "/")
EXTRACTED_INPUT = []
for _dirpath, _dirnames, _files in os.walk("/kaggle/input"):
    if "competitions" in _dirnames:
        _dirnames.remove("competitions")   # 26k RSNA DICOMs live there and no code
    if os.path.exists(os.path.join(_dirpath, CODE_MARKER)):
        EXTRACTED_INPUT.append(_dirpath)
        _dirnames[:] = []
repo_zip = next((p for p in CANDIDATES if os.path.exists(p)), None)

if repo_zip:
    print("using code archive:", repo_zip, f"({os.path.getsize(repo_zip):,} bytes)")
    shutil.rmtree(EXTRACT_TO, ignore_errors=True)
    os.makedirs(EXTRACT_TO, exist_ok=True)
    with zipfile.ZipFile(repo_zip) as zf:
        zf.extractall(EXTRACT_TO)
elif EXTRACTED_INPUT:
    # /kaggle/input is read-only and the cells below write results next to the code they import, so the
    # attached tree is copied rather than imported in place.
    _src_dir = EXTRACTED_INPUT[0].replace("/" + _MARKER_GLOB, "")
    print("Kaggle unzipped the attached code archive - copying", _src_dir, "->", EXTRACT_TO)
    shutil.rmtree(EXTRACT_TO, ignore_errors=True)
    shutil.copytree(_src_dir, EXTRACT_TO)
elif RUN_ON == "local":
    # Walk up from the notebook's directory until the marker file is found, so this works whether the
    # notebook is run from notebooks/ or from the repository root.
    EXTRACT_TO = os.getcwd()
    while EXTRACT_TO != os.path.dirname(EXTRACT_TO) and not os.path.exists(os.path.join(EXTRACT_TO, CODE_MARKER)):
        EXTRACT_TO = os.path.dirname(EXTRACT_TO)
    print("local run - using the working tree:", EXTRACT_TO)
else:
    raise FileNotFoundError(
        "repo.zip not found. Attach it to this kernel (Add Input > Files, or Settings > Google Drive and "
        "keep it in My Drive/aiml02/) - looked in: " + repr([p for p in CANDIDATES]))

matches = [dp for dp, _d, _f in os.walk(EXTRACT_TO) if os.path.exists(os.path.join(dp, CODE_MARKER))]
if not matches:
    raise RuntimeError(f"no {CODE_MARKER} under {EXTRACT_TO} - rezip the repository folder itself")
REPO_DIR = min(matches, key=lambda p: p.count(os.sep))
sys.path.insert(0, REPO_DIR)
print("repo at:", REPO_DIR)

from src.data.chest_dataset import ChestXrayDataset
from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD, gray_to_rgb, make_transform
from src.models.classifier import build_densenet121, load_checkpoint
from src.models.gradcam import (TARGET_LAYER, body_mask, gradcam, overlay, region_stats)
from src.data.dicom_io import to_uint8

print("imports from src/: dataset, transform, checkpoint loader, Grad-CAM - the code the tests cover")

In [ ]:
# notebook 03's artefacts. On Kaggle, attach the training kernel's output (Add Input > Your Code > the
# kernel) - its files land under /kaggle/input/<kernel-slug>/. On Colab, the Drive artifacts folder.
CK_SEARCH = sorted(glob.glob("/kaggle/input/**/densenet121_lungopacity.pt", recursive=True)) + [
    "/kaggle/working/densenet121_lungopacity.pt",
    "/content/drive/MyDrive/aiml02/artifacts/densenet121_lungopacity.pt",
    os.path.join(os.getcwd(), "artifacts", "densenet121_lungopacity.pt"),
    os.path.join(os.getcwd(), "..", "artifacts", "densenet121_lungopacity.pt"),
    os.path.join(os.getcwd(), "_nb03_scratch", "artifacts", "densenet121_lungopacity.pt"),
    os.path.join(REPO_DIR, "artifacts", "densenet121_lungopacity.pt"),
]
CK = next((p for p in CK_SEARCH if os.path.exists(p)), None)
if CK is None:
    found = sorted(glob.glob("/kaggle/input/**/*", recursive=True))[:40] if RUN_ON == "kaggle" else []
    raise FileNotFoundError(
        "densenet121_lungopacity.pt not found. Looked in: " + repr(CK_SEARCH) +
        "\nWhat is attached instead: " + repr(found) +
        "\nKaggle: run notebook 03 first, then Add Input > Your Code > that kernel (its outputs are "
        "this kernel's inputs). Colab: it is written to MyDrive/aiml02/artifacts/.")
print("checkpoint:", CK, f"({os.path.getsize(CK):,} bytes)")

MJ_SEARCH = sorted(glob.glob("/kaggle/input/**/densenet_metrics.json", recursive=True)) + [
    "/kaggle/working/densenet_metrics.json",
    os.path.join(REPO_DIR, "results", "densenet_metrics.json"),
    os.path.join(os.getcwd(), "..", "results", "densenet_metrics.json"),
    os.path.join(os.getcwd(), "_nb03_scratch", "results", "densenet_metrics.json"),
]
METRICS_JSON = next((p for p in MJ_SEARCH if os.path.exists(p)), None)
nb03 = json.load(open(METRICS_JSON, encoding="utf-8")) if METRICS_JSON else None
print("notebook 03 metrics:", METRICS_JSON or "absent - subgroup numbers below stand on their own")
if nb03:
    print("  host", nb03["runtime"].get("host"), "| test AUROC",
          nb03["evaluations"]["L3 DenseNet121 - TEST"]["auroc"], "| verdict:", nb03["verdict"])

In [ ]:
import pandas as pd

splits = {s: pd.read_csv(os.path.join(REPO_DIR, "results", f"split_{s}.csv"))
          for s in ("train", "val", "test")}
for s, f in splits.items():
    assert {"dicom_rel", "label", "patient_id"} <= set(f.columns), f"{s}: {list(f.columns)}"
overlap = set(splits["train"]["patient_id"]) & set(splits["test"]["patient_id"])
assert not overlap, f"{len(overlap)} patients in both train and test - the split CSVs are not the ones 03 used"
print("splits loaded:", {s: len(f) for s, f in splits.items()},
      "| train/test patient overlap:", len(overlap))
print("test opacity rate:", round(float(splits["test"]["label"].mean()), 4))

test = splits["test"].head(CONFIG["n_predict"]).reset_index(drop=True)
print("this notebook predicts on", len(test), "test images")

## 2. The trained model, and proof of which run it came from

`build_densenet121(pretrained=False)` rebuilds the architecture and `load_checkpoint` strict-loads
notebook 03's weights over it - so an architecture drift fails loudly here instead of producing a
plausible-looking but meaningless heat map.

In [ ]:
net = build_densenet121(num_classes=1, pretrained=False, repo_dir=REPO_DIR,
                        probe_size=CONFIG["train_size"])
ckpt = load_checkpoint(net, CK, map_location=device)
net.to(device).eval()

ck_cfg = ckpt.get("config", {})
print("checkpoint trained on:", ck_cfg.get("run_on"), "| epochs", ckpt.get("epoch"),
      "| threshold", ckpt.get("threshold"))
for key in ("train_size", "cache_size", "epochs_total", "batch_train", "seed"):
    print(f"  {key}: ckpt={ck_cfg.get(key)} nb04={CONFIG.get(key)}")
assert ck_cfg.get("train_size") == CONFIG["train_size"],     "notebook 04 would feed the model a different input size than it trained on"
assert ck_cfg.get("cache_size") == CONFIG["cache_size"], "the overlay would not be the model's input"
history = ckpt.get("history") or []
if history:
    best_row = max(history, key=lambda r: r["val_auroc"])
    print("best val AUROC from the checkpoint's own history:", best_row)

## 3. Decode the test images

The 448 px render cache is scratch: Kaggle wipes `/tmp` between kernels, so notebook 04 rebuilds it for
the images it actually needs instead of shipping 2.3 GB of JPEGs around as a kernel output.

In [ ]:
def dcm_count(folder):
    # os.scandir, never a shell glob: 26,684 names through globbing takes minutes on this filesystem.
    return sum(1 for e in os.scandir(folder) if e.name.endswith(".dcm")) if os.path.isdir(folder) else 0


images_dir = CONFIG["images_dir"]
z_search = sorted(glob.glob(f"/kaggle/input/**/{CONFIG['zip_name']}", recursive=True)) + [
    f"/content/drive/MyDrive/{CONFIG['zip_name']}",
    f"/content/drive/MyDrive/aiml02/{CONFIG['zip_name']}",
    os.path.join(REPO_DIR, "data", "raw", "rsna", CONFIG["zip_name"]),
]
extracted = sorted(p for p in
                   glob.glob("/kaggle/input/**/stage_2_train_images", recursive=True)
                   + [os.path.join(REPO_DIR, "data", "raw", "rsna", "stage_2_train_images")]
                   if dcm_count(p) >= 1000)

if dcm_count(images_dir) < 1000 and extracted:
    # The folder beats the archive when both are on disk: reading the pixels where they sit avoids a
    # second 3.3 GB extract, and it is the shape Kaggle's competition input ships today.
    images_dir = extracted[0]
    print("using extracted images folder:", images_dir,
          f"({dcm_count(images_dir):,} dcm) - unzip skipped")
elif dcm_count(images_dir) < 1000:
    zip_path = next((p for p in z_search if os.path.exists(p)), None)
    if zip_path is None:
        attached = {}
        if os.path.isdir("/kaggle/input"):
            for src in sorted(os.listdir("/kaggle/input")):
                top = os.path.join("/kaggle/input", src)
                attached[src] = sorted(os.listdir(top))[:8] if os.path.isdir(top) else "(file)"
        raise FileNotFoundError(
            f"{CONFIG['zip_name']} not found. Kaggle: Add Input > Competition > "
            "rsna-pneumonia-detection-challenge. Colab: upload it to My Drive. Looked in: "
            + repr(z_search) + f"\nWhat is actually attached under /kaggle/input: {attached}")
    print("unzipping", zip_path, f"({os.path.getsize(zip_path):,} bytes) - ~10 min")
    t0 = time.time()
    os.makedirs(images_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:      # 26,684 loose .dcm at the archive root, no inner folder
        zf.extractall(images_dir)
    print(f"unzipped in {time.time() - t0:.0f} s")

if dcm_count(images_dir) < 1000:
    for root, _dirs, _files in os.walk(images_dir):
        if dcm_count(root) >= 1000:
            images_dir = root
            break
n_dcm = dcm_count(images_dir)
print("DICOM files available:", f"{n_dcm:,}")
assert n_dcm >= 26000, f"expected 26,684 stage_2 images, found {n_dcm}"


In [ ]:
from concurrent.futures import ProcessPoolExecutor
from PIL import Image

CACHE_DIR = os.path.join(CONFIG["cache_root"], "test")
os.makedirs(CACHE_DIR, exist_ok=True)


def _render(task):
    # Import inside the worker: on Linux the pool forks and inherits pydicom, but spawn would not.
    import os
    from pydicom import dcmread
    from PIL import Image as _Image
    from src.data.dicom_io import to_uint8 as _to8
    src, dst, size = task
    if os.path.exists(dst):
        return "skip"
    try:
        img = _Image.fromarray(_to8(dcmread(src, force=True)))
    except Exception as exc:
        return f"error {os.path.basename(src)}: {type(exc).__name__}: {exc}"
    img = img.resize((size, size), _Image.BILINEAR).convert("L")
    img.save(dst, "JPEG", quality=90)
    return "ok"


tasks = [(os.path.join(images_dir, os.path.basename(r)),
          os.path.join(CACHE_DIR, os.path.basename(r).replace(".dcm", ".jpg")),
          CONFIG["cache_size"]) for r in test["dicom_rel"]]
t0, results = time.time(), []
def _drain(iterator):
    for i, res in enumerate(iterator, 1):
        results.append(res)
        if i % 500 == 0:
            el = time.time() - t0
            print(f"  rendered {i}/{len(tasks)} ({el:.0f} s, {el/i*1000:.0f} ms/file, "
                  f"eta {(len(tasks)-i)*el/i/60:.1f} min)", flush=True)


if os.name == "nt":
    # Windows spawn cannot re-import a function defined in a notebook cell; decode serially instead.
    _drain(map(_render, tasks))
else:
    workers = min(4, (os.cpu_count() or 2))
    with ProcessPoolExecutor(max_workers=workers) as pool:
        _drain(pool.map(_render, tasks, chunksize=32))
failed = [r for r in results if r.startswith("error")]
print(f"test: {len(results)-len(failed)} rendered, {len(failed)} failed in {time.time()-t0:.0f} s")
if failed:
    print("  first failures:", failed[:5])

## 4. Predictions on the test split

One forward pass, and the probabilities are written out - every fairness number later in this notebook
comes from that file, not from a second evaluation the reviewer cannot see.

In [ ]:
from torch.utils.data import DataLoader

ds = ChestXrayDataset(test, CACHE_DIR, make_transform(CONFIG["train_size"], train=False),
                      CONFIG["cache_size"])
print(f"{len(ds)} images | {int(ds.labels.sum())} positive | renders missing: {len(ds.missing_paths())}")
print("input pipeline:", make_transform(CONFIG["train_size"], train=False))
loader = DataLoader(ds, batch_size=CONFIG["batch_eval"], shuffle=False, num_workers=2,
                    pin_memory=(device == "cuda"))
net.eval()
probs, rows = [], []
t0 = time.time()
with torch.no_grad():
    for x, y in loader:
        logits = net(x.to(device, non_blocking=True)).squeeze(1)
        probs.extend(torch.sigmoid(logits).float().cpu().numpy().tolist())
        rows.extend(y.tolist())
probs = np.asarray(probs, dtype=np.float64)
labels = np.asarray(rows, dtype=np.float64)
print(f"inference on {len(probs):,} images in {(time.time()-t0)/60:.1f} min ({(time.time()-t0)/len(probs)*1000:.0f} ms/image)")

threshold = float(ckpt.get("threshold") or 0.5)
preds = (probs >= threshold).astype(int)
pred_df = test.copy()
pred_df["file_id"] = pred_df["dicom_rel"].map(lambda p: os.path.basename(p).replace(".dcm", ""))
pred_df["jpg"] = pred_df["file_id"] + ".jpg"
pred_df["prob"] = probs
pred_df["pred"] = preds
pred_df["correct"] = (preds == labels.astype(int)).astype(int)
print("threshold from notebook 03's validation Youden J:", threshold)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

overall = {"n": int(len(labels)), "prevalence": round(float(labels.mean()), 4),
           "auroc": round(float(roc_auc_score(labels, probs)), 4),
           "pr_auc": round(float(average_precision_score(labels, probs)), 4),
           "threshold": round(threshold, 4),
           "sensitivity": None, "specificity": None}
tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
overall.update({"sensitivity": round(tp / (tp + fn), 4) if tp + fn else None,
                "specificity": round(tn / (tn + fp), 4) if tn + fp else None,
                "confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}})
print(json.dumps(overall, indent=1))
if nb03:
    bar = nb03["bar_to_beat_l1_pixel_baseline_test_auroc"]
    print(f"L3 here {overall['auroc']:.4f} vs notebook 03's {nb03['evaluations']['L3 DenseNet121 - TEST']['auroc']:.4f} "
          f"| bar to beat {bar}")
    assert abs(overall["auroc"] - nb03["evaluations"]["L3 DenseNet121 - TEST"]["auroc"]) < 0.02, \
        "a different AUROC on the same checkpoint means the two notebooks preprocessed differently"

## 5. Grad-CAM: does it look at lungs?

`src/models/gradcam.py` hooks `features.denseblock4` (1024 channels, a 7x7 grid for a 224 px input) and
weights those channels by the gradient of the opacity logit. Two honest caveats that belong in front of
the pictures:

* **The map is 7x7 before upsampling.** After bilinear interpolation onto a 448 px render it *looks*
  smooth and is not. A 7x7 attribution cannot localise a nodule; it can only say roughly where the
  evidence was.
* **The overlay is drawn on the full render, but the model saw a 224 px centre crop** of a 234 px
  resize. That is a ~4 percent scale mismatch, not a re-projection - visible in the corner regions.

In [ ]:
def tensor_for(jpg_path):
    # One render -> the (1, 3, 224, 224) tensor the model was trained on, plus the raw 448 render.
    img = Image.open(os.path.join(CACHE_DIR, jpg_path)); img.load()
    grey = np.asarray(img, dtype=np.uint8)
    x = make_transform(CONFIG["train_size"], train=False)(gray_to_rgb(grey))[None]
    return x.to(device), grey


def attention_for(jpg_path):
    x, grey = tensor_for(jpg_path)
    hm = gradcam(net, x, target_layer=CONFIG["target_layer"], image_size=(CONFIG["cache_size"],) * 2)
    return hm, grey


# Pick the cases a reviewer should see, not the flattering ones: correct hits, missed opacities,
# confident false positives and true negatives.
cases = []
pos = pred_df[pred_df["label"] == 1]; neg = pred_df[pred_df["label"] == 0]
for tag, frame, n in [("hit", pos[pos["pred"] == 1].sort_values("prob", ascending=False), 3),
                      ("missed", pos[pos["pred"] == 0].sort_values("prob"), 3),
                      ("false positive", neg[neg["pred"] == 1].sort_values("prob", ascending=False), 2),
                      ("clear negative", neg[neg["pred"] == 0].sort_values("prob"), 1)]:
    for _, r in frame.head(n).iterrows():
        cases.append((tag, r))
print("gallery cases:", [c[0] for c in cases])

In [ ]:
import matplotlib
# Only force the headless backend where there is no inline one: Colab/Kaggle render %matplotlib inline,
# and calling use("agg") there blanks the figure.
if RUN_ON == "local":
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

cols = [c for c in cases]
ncol = 3
nrow = int(np.ceil(len(cols) / ncol))
fig, axes = plt.subplots(nrow, ncol * 2, figsize=(4.0 * ncol * 2, 4.0 * nrow), squeeze=False)
for j, (tag, r) in enumerate(cols):
    hm, grey = attention_for(r["jpg"])
    ov = overlay(grey, hm, alpha=0.5, threshold=0.3)
    ax = axes[j // ncol][ (j % ncol) * 2]
    ax.imshow(grey, cmap="gray"); ax.set_title(f"{tag}: label={int(r['label'])} p={r['prob']:.2f}", fontsize=9)
    ax.axis("off")
    ax = axes[j // ncol][(j % ncol) * 2 + 1]
    ax.imshow(ov); ax.set_title("Grad-CAM overlay", fontsize=8); ax.axis("off")
for j in range(len(cols), nrow * ncol):
    for k in range(ncol * 2):
        axes[j // ncol][k].axis("off")
fig.suptitle("Grad-CAM on the DenseNet121 lung-opacity classifier - education only, not a diagnosis", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.97))
png_path = os.path.join(CONFIG["out_dir"], "gradcam_gallery.png")
fig.savefig(png_path, dpi=110, bbox_inches="tight")
plt.show()
print("saved", png_path, f"({os.path.getsize(png_path):,} bytes)")

In [ ]:
# The same question quantitatively, because a gallery is anecdote. `in_body_fraction` asks what share of
# the attention mass falls inside the Otsu-thresholded body silhouette: attention on a corner stamp or
# on the air outside the chest scores low.
focus_rows = []
sample = pred_df.iloc[np.linspace(0, len(pred_df) - 1, min(CONFIG["n_focus"], len(pred_df))).astype(int)]
t0 = time.time()
for i, (_, r) in enumerate(sample.iterrows(), 1):
    hm, grey = attention_for(r["jpg"])
    st = region_stats(hm, body_mask(grey))
    peak = st["peak"]
    focus_rows.append({"file_id": r["file_id"], "label": int(r["label"]), "pred": int(r["pred"]),
                       "prob": round(float(r["prob"]), 4),
                       "in_body_fraction": st["in_body_fraction"], "lateral_bias": st["lateral_bias"],
                       "peak_row": None if peak is None else peak[0],
                       "peak_col": None if peak is None else peak[1],
                       "centroid_row": (st["centroid"] or (None, None))[0],
                       "centroid_col": (st["centroid"] or (None, None))[1],
                       "mass": round(float(hm.sum()), 3)})
    if i % 25 == 0:
        print(f"  {i}/{len(sample)} maps ({time.time()-t0:.0f} s)", flush=True)
focus = pd.DataFrame(focus_rows)
focus_path = os.path.join(CONFIG["out_dir"], "gradcam_focus.csv")
focus.to_csv(focus_path, index=False)
print(f"{len(focus)} maps in {(time.time()-t0)/60:.1f} min -> {focus_path}")

print(focus.groupby("label")[["in_body_fraction", "lateral_bias", "centroid_row"]].agg(["mean", "median", "count"]))
low = focus[focus["in_body_fraction"] < 0.5]
print(f"images with under half the attention inside the body: {len(low)} / {len(focus)}")
if len(low):
    print("  worst 5:", low.nsmallest(5, "in_body_fraction")[["file_id", "label", "prob", "in_body_fraction"]].to_string(index=False))

## 6. Fairness: does it work equally well for everyone?

Subgroups come from the DICOM headers (`PatientSex`, `PatientAge`) - the demographic fields RSNA left
populated - reaching this notebook either through the split CSVs or through the cached header scan, and the cell
below prints which one it found. AUROC per group with a bootstrap 95% CI, because a group of 40 images
can move 0.05 on noise alone and a table without intervals invites exactly that mistake.

A fourth slice is not demographic: the radiological class sitting behind the binary label. The negatives are
`Normal` and `No Lung Opacity / Not Normal`, which are different clinical groups, so a model can score well by
learning "obviously abnormal" rather than "opacity". Splitting on that class is the cheapest way to notice.

In [ ]:
import glob as _glob

# Demographics come from the richest source that is actually reachable:
#  1. tools/scan_dicom_headers.py's cached CSV over all 26,684 files - it also has PatientPosition and
#     the window tags, which the two sources below do not carry;
#  2. the split CSVs themselves: notebook 01 pulled sex + age out of the headers once, and those CSVs
#     ship inside repo.zip, so this kernel can answer the fairness questions without opening a DICOM.
HEADERS_HUNT = [CONFIG["headers_csv"]] + sorted(_glob.glob("/kaggle/input/**/dicom_headers.csv", recursive=True)) + [
    "/content/drive/MyDrive/aiml02/dicom_headers.csv",
    os.path.join(REPO_DIR, "data", "raw", "rsna", "dicom_headers.csv"),
]
headers_csv = next((p for p in HEADERS_HUNT if p and os.path.exists(p)), None)

if headers_csv:
    h = pd.read_csv(headers_csv, usecols=["file_id", "PatientSex", "PatientAge", "PatientPosition",
                                          "WindowCenter", "WindowWidth"])
    h = h[h["file_id"].isin(pred_df["file_id"])]
    print("headers from the cached scan:", headers_csv, f"({len(h):,} of the predicted images)")
    used = "cached_scan"
elif {"PatientSex", "age"} <= set(pred_df.columns):
    h = pred_df[["file_id", "PatientSex", "age"]].rename(columns={"age": "PatientAge"}).copy()
    print("demographics from the split CSVs (notebook 01 read the headers once) - no DICOM file opened")
    used = "split_csv"
else:
    raise KeyError(
        "neither the cached header scan nor demographics inside the split CSVs. Run tools/"
        "scan_dicom_headers.py (or notebook 01, which writes sex + age into split_*.csv) first - "
        "looked for a scan in: " + repr(HEADERS_HUNT))

# pred_df can already carry the same columns (the split CSVs do). Merging two copies of PatientSex would
# silently rename them PatientSex_x / PatientSex_y, so `h` is the single source from here on.
meta = pred_df.drop(columns=[c for c in ("PatientSex", "PatientAge", "age")
                             if c in pred_df.columns]).merge(h, on="file_id", how="left")
meta["age"] = pd.to_numeric(meta["PatientAge"], errors="coerce")
meta["sex"] = meta["PatientSex"].astype("string").str.strip().str.upper().replace({"": pd.NA})


def band(a):
    if pd.isna(a): return "unknown"
    for lo, hi in CONFIG["age_bands"]:
        if lo <= a < hi:
            # no meaningful ceiling in this archive (highest recorded age is 155), so 80+ reads better
            # than 80-199 and matches the model card row
            return f"{lo}+" if (lo, hi) == CONFIG["age_bands"][-1] else f"{lo}-{hi - 1}"
    return f"{CONFIG['age_bands'][-1][0]}+"


meta["age_band"] = meta["age"].map(band)
print("joined demographic metadata via", used, "| rows", len(meta),
      "| sex known:", int(meta["sex"].notna().sum()), "| age known:", int(meta["age"].notna().sum()))
# The radiological class behind each binary label: negatives split into a clear chest and an abnormal
# chest without opacity, which are different clinical groups and behave differently.
CLASS_HUNT = sorted(_glob.glob("/kaggle/input/**/stage_2_detailed_class_info.csv", recursive=True)) + [
    os.path.join(os.path.dirname(CONFIG["images_dir"]), "stage_2_detailed_class_info.csv"),
    os.path.join(_LOCAL_ROOT, "data", "raw", "rsna", "stage_2_detailed_class_info.csv"),
    os.path.join(REPO_DIR, "data", "raw", "rsna", "stage_2_detailed_class_info.csv"),
]
class_csv = next((p for p in CLASS_HUNT if os.path.exists(p)), None)
if class_csv:
    ci = pd.read_csv(class_csv)
    # stage_2 keys the image on a column actually named patientId (one uuid per picture, no study id);
    # later RSNA releases use dicom_id / ImageID / SeriesInstanceUID. Same detection as notebook 01.
    idcol = next((c for c in ("patientId", "dicom_id", "ImageID", "SeriesInstanceUID")
                  if c in ci.columns), None)
    if idcol is None or "class" not in ci.columns:
        raise KeyError(f"{class_csv} has neither an image id nor a class column: {list(ci.columns)}")
    ci = ci.drop_duplicates(idcol)
    ci = ci[ci[idcol].isin(pred_df["file_id"])]
    meta = meta.merge(ci[[idcol, "class"]].rename(columns={idcol: "file_id", "class": "class_source"}),
                      on="file_id", how="left")
    print("label source from:", class_csv, f"| id column {idcol!r} | "
      f"{meta['class_source'].notna().sum()} of {len(meta)} matched")
else:
    meta["class_source"] = "unavailable"
    print("stage_2_detailed_class_info.csv not reachable - the label-source slice will have one row saying so")
meta["class_source"] = meta["class_source"].fillna("unavailable")

meta.to_csv(os.path.join(CONFIG["out_dir"], "test_predictions_with_metadata.csv"), index=False)


In [ ]:
from sklearn.metrics import roc_auc_score as _auc


def auroc_ci(y, p, reps=CONFIG["bootstrap"], seed=CONFIG["seed"], alpha=0.05):
    # AUROC with a percentile bootstrap CI; None when the group has only one class present.
    y = np.asarray(y); p = np.asarray(p)
    if len(y) < 5 or len(np.unique(y)) < 2:
        return None
    rng = np.random.default_rng(seed)
    point = float(_auc(y, p))
    boots = []
    for _ in range(reps):
        idx = rng.integers(0, len(y), len(y))
        yy, pp = y[idx], p[idx]
        if len(np.unique(yy)) < 2:
            continue
        boots.append(float(_auc(yy, pp)))
    if not boots:
        return None
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return {"n": int(len(y)), "positives": int(y.sum()), "auroc": round(point, 4),
            "ci95": [round(float(lo), 4), round(float(hi), 4)], "bootstraps": len(boots)}


def subgroup_table(frame, by):
    rows = []
    for key, g in frame.groupby(by, dropna=False):
        entry = auroc_ci(g["label"].to_numpy(), g["prob"].to_numpy())
        focus_g = focus[focus["file_id"].isin(g["file_id"])] if len(focus) else pd.DataFrame()
        rows.append({"group": str(key), "n": len(g), "positives": int(g["label"].sum()),
                     "opacity_rate": round(float(g["label"].mean()), 4),
                     "auroc": None if entry is None else entry["auroc"],
                     "ci95": None if entry is None else entry["ci95"],
                     "sens_at_thr": round(float(g.loc[g["label"] == 1, "pred"].mean()), 4) if (g["label"] == 1).any() else None,
                     "spec_at_thr": round(float(1 - g.loc[g["label"] == 0, "pred"].mean()), 4) if (g["label"] == 0).any() else None,
                     "median_in_body_fraction": None if focus_g.empty else round(float(focus_g["in_body_fraction"].median()), 4)})
    return pd.DataFrame(rows)


tables = {"sex": subgroup_table(meta, "sex"), "age_band": subgroup_table(meta, "age_band"),
          "class_source": subgroup_table(meta, "class_source"),
          "sex_x_age": subgroup_table(meta, ["sex", "age_band"])}
for name, t in tables.items():
    print(f"\n--- {name} ---")
    print(t.to_string(index=False))
all_auc = auroc_ci(meta["label"].to_numpy(), meta["prob"].to_numpy())
print("\noverall:", json.dumps(all_auc))

gaps = {}
for name, t in tables.items():
    v = t["auroc"].dropna()
    if len(v) >= 2:
        gaps[name] = {"max_gap": round(float(v.max() - v.min()), 4),
                      "best": str(t.loc[v.idxmax(), "group"]), "worst": str(t.loc[v.idxmin(), "group"])}
print("largest AUROC gap between subgroups:", json.dumps(gaps, indent=1))
print("Note: a gap in *measured* performance is not proof of bias in the model - it can also come from\n"
      "differing prevalence, camera protocol or case mix within a group. Report it, do not over-read it.")

## 7. Missing data, and what the pipeline did about it

Every count below is measured on the files that were actually loaded, not copied from the dataset
paper. The policy column is the part a reviewer can audit: absence is handled by a documented rule, not
by a silent drop.

In [ ]:
audit = []

def add(item, count, total, policy):
    audit.append({"item": item, "n_missing": int(count), "n_total": int(total),
                  "pct_missing": round(100.0 * count / total, 2) if total else None, "policy": policy})


add("test images with no 448px render in the cache", len(ds.missing_paths()), len(ds),
    "rendered as an all-black frame so one failed decode cannot lose an epoch; the path is kept in densenet_metrics.json")
add("PatientSex empty or unreadable", int(meta["sex"].isna().sum()), len(meta),
    "kept in the predictions, grouped as 'unknown', excluded from the sex table's gap arithmetic")
add("PatientAge empty or unreadable", int(meta["age"].isna().sum()), len(meta),
    "same: an 'unknown' age band, never imputed by the cohort median")
if "PatientPosition" in h.columns:
    add("PatientPosition empty", int(h["PatientPosition"].isna().sum() + (h["PatientPosition"].astype(str).str.strip() == "").sum()), len(h),
        "not used as a model input; verified empty for this whole archive, so AP/PA conditioning is impossible here")
if "WindowCenter" in h.columns:
    wc_missing = int(h["WindowCenter"].isna().sum() + (h["WindowCenter"].astype(str).str.strip() == "").sum())
    add("WindowCenter / WindowWidth tags absent", wc_missing, len(h),
        "src/data/dicom_io.py falls back to C=1024 W=4096 - and to the 8-bit secondary-capture branch when BitsAllocated<=8, which is the case for these CR files")
add("test images whose radiological class could not be joined",
    int((meta["class_source"] == "unavailable").sum()), len(meta),
    "the label-source slice then reports one 'unavailable' group instead of guessing a class; the binary "
    "label itself never came from this file, so no prediction depends on it")
dups = int(meta["file_id"].duplicated().sum())
add("duplicate image paths in the predicted set", dups, len(meta), "would be a rows-vs-images leak; notebook 01 splits by patient, not by row")
dup_pat = int(splits["test"]["patient_id"].duplicated().sum())
add("test images sharing a PatientID with another test image", dup_pat, len(splits["test"]),
    "allowed: a patient can contribute several films; splits are held apart by PatientID, which the cell above asserts")

audit_df = pd.DataFrame(audit)
print(audit_df.to_string(index=False))
audit_df.to_csv(os.path.join(CONFIG["out_dir"], "missing_data_audit.csv"), index=False)

## 8. Summary written out, and the honest reading of it

In [ ]:
import math


def json_safe(o):
    """NaN/Inf are not JSON. A subgroup table always has empty cells, so sweep them to None."""
    if isinstance(o, float) and not math.isfinite(o):
        return None
    if isinstance(o, dict):
        return {k: json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [json_safe(v) for v in o]
    return o


summary = {
    "generated": time.strftime("%Y-%m-%d %H:%M"),
    "host": {"run_on": CONFIG["run_on"], "device": device, "torch": torch.__version__,
             "python": platform.python_version(), "seed": CONFIG["seed"]},
    "checkpoint": {"path": CK, "epoch": ckpt.get("epoch"), "threshold": threshold,
                   "trained_on_run_on": ck_cfg.get("run_on"), "config": ck_cfg},
    "notebook_03_metrics": None if nb03 is None else {
        "test_auroc": nb03["evaluations"]["L3 DenseNet121 - TEST"]["auroc"],
        "bar_to_beat": nb03["bar_to_beat_l1_pixel_baseline_test_auroc"], "verdict": nb03["verdict"]},
    "test_subset": overall,
    "preprocessing": {"cache_size": CONFIG["cache_size"], "model_input": CONFIG["train_size"],
                      "normalisation_mean": IMAGENET_MEAN, "normalisation_std": IMAGENET_STD,
                      "windowing": "src/data/dicom_io.py:to_uint8"},
    "gradcam": {"target_layer": CONFIG["target_layer"], "maps": int(len(focus)),
                "median_in_body_fraction": None if focus.empty else round(float(focus["in_body_fraction"].median()), 4),
                "maps_with_under_half_in_body": int((focus["in_body_fraction"] < 0.5).sum()) if len(focus) else None,
                "by_label": {str(k): round(float(v), 4) for k, v in focus.groupby("label")["in_body_fraction"].median().items()} if len(focus) else {},
                "resolution_note": "7x7 grid upsampled to 448px; overlay is on the uncropped render while the model saw a 224px centre crop"},
    "fairness": {name: {"table": t.to_dict(orient="records"), "max_gap": gaps.get(name)}
                 for name, t in tables.items()},
    "metadata_source": used,
    "missing_data": audit_df.to_dict(orient="records"),
    "limitations": [
        "Grad-CAM attributes a decision to image regions; it does not identify a finding and is not localisation ground truth.",
        "Single-institution public challenge data; sex and age come from headers, not from any clinical annotation.",
        "Subgroup gaps are measured differences, not proven model bias - case mix and prevalence differ by group.",
        "PatientPosition is empty across this archive, so AP/PA projection effects cannot be conditioned on or tested.",
        "For education only. Not for clinical use. No regulatory clearance.",
    ],
    "artifacts": {"gallery_png": os.path.join(CONFIG["out_dir"], "gradcam_gallery.png"),
                  "focus_csv": focus_path,
                  "predictions_csv": os.path.join(CONFIG["out_dir"], "test_predictions_with_metadata.csv"),
                  "audit_csv": os.path.join(CONFIG["out_dir"], "missing_data_audit.csv")},
}
out_json = os.path.join(CONFIG["out_dir"], "gradcam_fairness_summary.json")
with open(out_json, "w", encoding="utf-8") as fh:
    json.dump(json_safe(summary), fh, indent=1, default=str, allow_nan=False)
print("wrote", out_json)
print(json.dumps(json_safe({k: summary[k] for k in ("host", "test_subset", "gradcam", "metadata_source")}),
                 indent=1, default=str, allow_nan=False))

## 9. Reading the result honestly

* A **high `in_body_fraction`** is necessary, not sufficient: attention inside the chest can still be
  tracking ribs, the diaphragm edge, or brightness - the shortcut notebook 02 measured.
* A **missed opacity whose map still points at the lung** is a different failure from one whose map is
  elsewhere: the first is a modelling/threshold problem, the second says the evidence is not visual.
* **Subgroup gaps with overlapping bootstrap intervals are not findings.** The table prints the
  intervals so the reader can check that rather than take the point estimates on trust.
* **Nothing here is validated for clinical use.** RSNA stage_2 labels are a surrogate (opacity present
  or not), not a radiologist's diagnosis, and this model has never been through a prospective review.

Next: `docs/model_card.md` takes these tables verbatim; the FastAPI service in `src/api/` returns the
same overlay for a single study, and the React viewer shows it with Approve / Reject.